# Week 5 — LoRA Adapter Training (Colab) — traceable record

Trains a precision-targeted **QLoRA** adapter on `qwen2.5:3b` for the ClimateClaimVerifier
claim/opinion classifier, evaluates it on the 100-row eval set, and exports it to **GGUF**
for the Streamlit "Base vs Adapter" demo.

**Verdict (recorded in `project_description.md`):** the adapter shifts the precision/recall
frontier but **no** training balance reaches recall ≥ 0.90 *and* precision ≥ 0.85 together,
so production ships the **base** classifier (recall-first) and this adapter is **demo-only**.
The **40% claim** balance used below is the compromise kept for the demo — see
*Experimentation* at the end for the full ratio sweep that led to it.

Run top-to-bottom for the final 40% adapter. Requires a GPU runtime (Runtime → Change runtime type → T4).

## 1 · Setup

In [ ]:
!nvidia-smi -L

In [ ]:
!git clone https://github.com/Sadaf-Burhan/ClimateClaimVerifier.git
%cd ClimateClaimVerifier
!git pull              # latest scripts (build_lora_trainset.py with caching, lora_seed.jsonl)
!mkdir -p tmp/colab

In [ ]:
# Upload tmp/colab/train_candidates.jsonl (produced locally by `build_lora_trainset.py select`)
from google.colab import files
import shutil
files.upload()
shutil.move("train_candidates.jsonl", "tmp/colab/train_candidates.jsonl")
!wc -l tmp/colab/train_candidates.jsonl     # expect 98

In [ ]:
# Ollama + teacher model (must be STRONGER than the 3B student — gemma2:2b was too weak)
import subprocess, time
!curl -fsSL https://ollama.com/install.sh | sh
subprocess.Popen(["ollama", "serve"]); time.sleep(5)
!ollama pull qwen2.5:7b
!pip -q install ollama pyyaml

## 2 · Data generation — teacher labels + manual corrections → master

The hand-labelled seed (`data/lora_seed.jsonl`) covers the hard conspiracy boundary; the
teacher labels the clear bulk. We then apply the Q5 manual corrections (news-headlines the
teacher demoted to opinion) and freeze a **master** so ratio tuning never re-calls the teacher.

In [ ]:
!python scripts/build_lora_trainset.py label --teacher qwen2.5:7b
import json
rows = [json.loads(l) for l in open("tmp/colab/lora_trainset.jsonl")]
# Q5 correction: these news-headline-with-commentary posts contain a checkable named
# event/source — flip teacher's 'opinion' back to 'claim' (protects recall).
for r in rows:
    if any(t in r["text"] for t in ["Suffolk Reform council", "Trump uses wartime powers", "Local researchers test a new method"]):
        r["has_claim"] = True; r["reason"] = "Specific named event/source — checkable"
with open("tmp/colab/lora_full.jsonl", "w") as f:        # MASTER — never overwritten
    for r in rows: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(sum(r["has_claim"] for r in rows), "claim /", sum(not r["has_claim"] for r in rows), "opinion -> master saved")

### Q5 validation — review the labels before training

In [ ]:
import json, re
from collections import Counter
rows = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
print("balance:", Counter(r["has_claim"] for r in rows))
signal = re.compile(r"\d|°|noaa|nasa|ipcc|environment canada|\brecord\b|%|\bmm\b|\bcm\b|degrees|category\s*\d", re.I)
print("\n-- OPINION-labelled but has a checkable signal (review: should any be CLAIM?) --")
for i, r in enumerate(rows):
    if not r["has_claim"] and signal.search(r["text"]):
        print(i, "|", r["text"][:80])

In [ ]:
import json, random
rows = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
for r in random.sample(rows, 15):
    print(r["has_claim"], "|", r["text"][:65], "->", r["reason"])

## 3 · Build training set @ 40% claim (chosen balance)

In [ ]:
import json, random
full = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
cl = [r for r in full if r["has_claim"]]; op = [r for r in full if not r["has_claim"]]
random.seed(42); random.shuffle(op)
CLAIM_FRAC = 0.40
train = cl + op[:round(len(cl) * (1 - CLAIM_FRAC) / CLAIM_FRAC)]; random.shuffle(train)
with open("tmp/colab/lora_trainset.jsonl", "w") as f:
    for r in train: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print(len(train), "examples,", sum(r["has_claim"] for r in train), "claim /", sum(not r["has_claim"] for r in train), "opinion")

## 4 · Train — QLoRA (Unsloth)

Uses `trl.SFTConfig` (not `TrainingArguments`) to avoid the SFTConfig pickling error, and a FRESH base each run.

In [ ]:
!pip -q install unsloth
from unsloth import FastLanguageModel
from datasets import Dataset
from trl import SFTTrainer, SFTConfig
import json

model, tokenizer = FastLanguageModel.from_pretrained(
    "unsloth/Qwen2.5-3B-Instruct-bnb-4bit", max_seq_length=1024, load_in_4bit=True)
model = FastLanguageModel.get_peft_model(
    model, r=8, lora_alpha=16, lora_dropout=0,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing="unsloth", random_state=42)

SYSTEM = ("You are a climate claim detector. Decide whether a post contains a verifiable "
  "factual claim about a climate or extreme weather event. A CLAIM has a specific checkable "
  "assertion — a named event, measurement, official source, date, place, or mechanism — "
  "checkable even if false; a checkable fact embedded in a rant is still a CLAIM. An OPINION "
  "has nothing checkable: feelings, jokes, vague conspiracy accusations, personal experiences, "
  "or general viewpoints. First note in `thought` what is checkable, then decide. JSON only.")

def to_text(r):
    tgt = json.dumps({"thought": r.get("thought",""), "has_claim": r["has_claim"], "reason": r.get("reason","")}, ensure_ascii=False)
    return tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM},
         {"role":"user","content":f'Post: "{r["text"][:500]}"'},
         {"role":"assistant","content":tgt}], tokenize=False)

rows = [json.loads(l) for l in open("tmp/colab/lora_trainset.jsonl")]
ds = Dataset.from_dict({"text": [to_text(r) for r in rows]})
SFTTrainer(model=model, tokenizer=tokenizer, train_dataset=ds,
    args=SFTConfig(dataset_text_field="text", max_seq_length=1024,
        per_device_train_batch_size=2, gradient_accumulation_steps=4,
        warmup_steps=5, num_train_epochs=3, learning_rate=2e-4,
        logging_steps=5, optim="adamw_8bit", seed=42, output_dir="outputs",
        save_strategy="no")).train()
model.save_pretrained("climate_claim_lora")
print("adapter trained @ 40% claim and saved -> climate_claim_lora/")

## 5 · Evaluate the adapter on the 100-row eval set

Served on the lean prompt it was trained on. Base reference (8-shot Ollama, single-post) ≈ recall 0.938 / precision 0.703. Target: recall ≥ 0.90 AND precision ≥ 0.85; the 50k-acres post must stay `claim`.

In [ ]:
import json, re, csv
from unsloth import FastLanguageModel
FastLanguageModel.for_inference(model)
eval_rows = list(csv.DictReader(open("data/claim_eval.csv", encoding="utf-8")))

def classify(text):
    p = tokenizer.apply_chat_template(
        [{"role":"system","content":SYSTEM}, {"role":"user","content":f'Post: "{text[:500]}"'}],
        tokenize=False, add_generation_prompt=True)
    ins = tokenizer(p, return_tensors="pt").to("cuda")
    out = model.generate(**ins, max_new_tokens=120, do_sample=False)
    txt = tokenizer.decode(out[0][ins.input_ids.shape[1]:], skip_special_tokens=True)
    m = re.search(r"\{.*\}", txt, re.DOTALL)
    try: return "claim" if (m and json.loads(m.group()).get("has_claim")) else "opinion"
    except: return "opinion"

pr = [classify(r["post_text"]) for r in eval_rows]
tp = sum(a=="claim" and r["expected_label"]=="claim" for r,a in zip(eval_rows,pr))
fn = sum(a=="opinion" and r["expected_label"]=="claim" for r,a in zip(eval_rows,pr))
fp = sum(a=="claim" and r["expected_label"]=="opinion" for r,a in zip(eval_rows,pr))
print(f"ADAPTER @40%: recall={tp/(tp+fn or 1):.3f}  precision={tp/(tp+fp or 1):.3f}  FN={fn}  FP={fp}")
print("50k test:", classify("50,000 acres of ice sheets have melted, what a lie these governments are trying to convince people of Alaska"))

## 6 · Export to GGUF (for the Streamlit demo)

Unsloth merges the adapter, converts via llama.cpp, and quantizes in one call. Download, then register locally: `ollama create qwen2.5-3b-claim-lora -f models/Modelfile`.

In [ ]:
model.save_pretrained_gguf("climate_claim_gguf", tokenizer, quantization_method="q4_k_m")
import glob, os
from google.colab import files
g = glob.glob("climate_claim_gguf/*.gguf")[0]
print("GGUF:", g, f"({os.path.getsize(g)/1e9:.1f} GB)")
files.download(g)

---
## Experimentation — the ratio sweep that led to 40%

Class balance is a **dial** that trades recall against precision. Holding everything else
constant, three training balances were measured on the same eval:

| training balance | recall | precision | note |
|---|---|---|---|
| ~26% claim (full, opinion-heavy) | 0.812 | 0.951 | overshot → precision, recall fails |
| 50% claim (balanced) | 0.958 | 0.742 | overshot → recall, precision barely > base |
| **40% claim (chosen)** | 0.854 | 0.788 | the compromise kept for the demo |

No balance reaches recall ≥ 0.90 **and** precision ≥ 0.85 together — the measured finding
behind the "ship base, no adapter" verdict.

The cells below reproduce the other two balances. **They overwrite `lora_trainset.jsonl` and,
after a retrain, the 40% adapter** — run only to reproduce the sweep, then re-run sections 3–6
to restore the 40% adapter.

> Historical note: the first training run used `transformers.TrainingArguments`, which raised
> a `PicklingError` on `SFTConfig` at the final save (the adapter weights still wrote to the
> checkpoint). The fix is `trl.SFTConfig` with `save_strategy="no"`, used in section 4.

In [ ]:
# EXPERIMENT — 50/50 balance (then re-run the Train + Eval cells in sections 4-5)
import json, random
full = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
cl = [r for r in full if r["has_claim"]]; op = [r for r in full if not r["has_claim"]]
random.seed(42); random.shuffle(op)
train = cl + op[:len(cl)]; random.shuffle(train)        # 50% claim
with open("tmp/colab/lora_trainset.jsonl", "w") as f:
    for r in train: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("50/50:", len(train), "examples,", sum(r["has_claim"] for r in train), "claim   -> measured 0.958 / 0.742")

In [ ]:
# EXPERIMENT — full opinion-heavy set, ~26% claim (then re-run Train + Eval)
import json, random
full = [json.loads(l) for l in open("tmp/colab/lora_full.jsonl")]
random.seed(42); random.shuffle(full)
with open("tmp/colab/lora_trainset.jsonl", "w") as f:
    for r in full: f.write(json.dumps(r, ensure_ascii=False) + "\n")
print("~26% claim:", len(full), "examples,", sum(r["has_claim"] for r in full), "claim   -> measured 0.812 / 0.951")